[![Open in GitHub Codespaces](https://github.com/codespaces/badge.svg)](https://codespaces.new/adairgranados/adair-granados?quickstart=1)


<!-- gen:values.yml#weeks.06.notebook-title|Debrief -->
# Week 6 Debrief: DataViz I: Visual encoding and chart choice
<!-- /gen -->

*This is for [Quest](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-quest) 06: The Chart Doctor, and this week's [AI failure gallery](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-ai-failure-gallery). As always, read it **after** you have attempted the quest.*

**How to read this notebook.** Each deck chart gets the following:

- A model one-line [TRACE](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-trace) verdict, the [Bronze](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-bronze) standard
- The reason for that verdict
- The direction you could give your [Associate](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-associate)
- A rebuilt chart

Compare each verdict and rebuild with yours. Note the differences in your [QuestLog](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-questlog). These are example answers. Any rebuild that passes TRACE, and that you can explain, is a good answer.

Run the data-setup cell below first.

In [ ]:
# --- Data setup: Run this cell first (runs anywhere, no download needed) ---
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt

rng = np.random.default_rng(5813)  # module seed: Do not change

# Beans of Egham's sites. The first five are the flagships.
SITES = ["Egham High St", "Staines Bridge", "Windsor Yard", "Runnymede Park",
         "Virginia Water", "Chertsey Lane", "Addlestone Sq", "Weybridge Stn",
         "Sunbury Cross", "Shepperton Lock", "Ashford Common", "Feltham Green"]
FLAGSHIPS = SITES[:5]
# The twelve months to Oct 2026, the year the deck covers
PERIODS = pd.period_range("2025-11", "2026-10", freq="M")
MONTHS = PERIODS.strftime("%b")

# (1) Revenue by menu item over the twelve months (£), drinks only.
#     Food and retail beans are in the category table (3).
menu_rev = pd.Series(
    {"Flat white": 237_100, "Latte": 183_700, "Cappuccino": 125_800, "Iced latte": 115_000,
     "Americano": 70_500, "Mocha": 50_300, "Hot chocolate": 43_400, "Tea": 35_900,
     "Espresso": 28_500, "Filter": 16_000, "Babyccino": 5_300},
    name="twelve_month_revenue_gbp")

# (2) Average transaction value at the five flagship sites (£)
atv = pd.Series({"Egham High St": 5.02, "Staines Bridge": 4.98, "Windsor Yard": 4.93,
                 "Runnymede Park": 4.96, "Virginia Water": 5.00}, name="avg_transaction_gbp")

# (3) Monthly revenue by product category (£k), twelve months to Oct 2026
season = 1 + 0.35 * np.sin((PERIODS.month - 4) / 12 * 2 * np.pi)        # peaks in high summer
lift = np.clip(season - 1, 0, None)                                     # summer lift only
monthly_cat = pd.DataFrame({
    "Espresso drinks": 44 + rng.normal(0, 1.0, 12),
    "Iced drinks":     7 + np.linspace(0, 12, 12) + 17 * lift + rng.normal(0, 1.0, 12),
    "Food":            25.5 + rng.normal(0, 1.0, 12),
    "Tea":             7.6 + rng.normal(0, 0.7, 12),
    "Hot chocolate":   9.4 - 5 * lift + rng.normal(0, 0.7, 12),
    "Retail beans":    5.1 + rng.normal(0, 0.5, 12),
}, index=MONTHS).round(1)

# (4) Monthly paid-social spend (£k). It rises through the year, as revenue does
marketing = pd.Series(np.round(1.5 + 0.12 * np.arange(12) + 5 * lift + rng.normal(0, 0.1, 12), 2),
                      index=MONTHS, name="paid_social_gbp_k")

# (5) Quarterly revenue (£k), 2024–2026. Q4 2026 is the founder's forecast for the pitch
quarterly = pd.DataFrame({"2024": [212, 231, 268, 240],
                          "2025": [246, 270, 315, 279],
                          "2026": [285, 312, 364, 322]}, index=["Q1", "Q2", "Q3", "Q4"])

# (6) Weekly transactions per site, the 52 trading weeks to late Oct 2026
weeks = np.arange(1, 53)
base = np.array([650, 530, 575, 380, 425, 350, 320, 395, 300, 270, 310, 380])
trend = np.zeros((12, 52)); trend[-1] = -np.linspace(0, 130, 52)
seasonal = 35 * np.sin((weeks - 8) / 52 * 2 * np.pi)
weekly_sites = pd.DataFrame(
    (base[:, None] + seasonal[None, :] + trend + rng.normal(0, 17, (12, 52))).round().astype(int).T,
    index=pd.Index(weeks, name="week"), columns=SITES)

print(f"Created (in memory, seed 5813): menu_rev ({len(menu_rev)} items) · atv ({len(atv)} sites) · "
      f"monthly_cat {monthly_cat.shape} · marketing ({len(marketing)} months) · "
      f"quarterly {quarterly.shape} · weekly_sites {weekly_sites.shape}")

## Diagnoses and rebuilds

One fault appears twice, in different forms: The axes suggest a pattern that the data does not show. Chart 2 truncates its baseline, and chart 4 uses twin y-axes. Both are **C** fails.

### Chart 1: The crowded pie

> **Verdict: T fail (with R problems).** A slice for every menu item makes the reader compare angles to answer a *ranking* question. Investors need to see at a glance which drinks sell. Rebuild it as a sorted horizontal bar chart, with shares labelled.

The reader's task is to rank the menu. Ranking needs position on a common scale, the top of the encoding ladder. A pie gives angle, the third rung. Past three or four slices, it is also hard to tell which label belongs to which slice, and the labels of the smallest slices overlap. The pie also hides the deck's best fact: Three drinks bring in 60% of revenue.

**The direction:** *"Draw a sorted horizontal bar of menu revenue share. Colour the top three #0072B2 and the rest grey. Label each bar with its percentage, no legend. Add a takeaway title with the top-three share computed from the data, and a source line."*

In [ ]:
# Rebuild 1: The menu ranked by share of revenue
share = (menu_rev / menu_rev.sum() * 100).sort_values()
top3 = share.nlargest(3)
colours = ["#0072B2" if item in top3.index else "#949494" for item in share.index]

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.barh(share.index, share, color=colours)
ax.bar_label(bars, fmt="{:.0f}%", padding=3, fontsize=9)

ax.set_title(f"Three drinks bring in {top3.sum():.0f}% of revenue",
             loc="left", fontweight="bold")
ax.set_xlabel("Share of twelve-month revenue (%)")
ax.set_xlim(0, share.max() * 1.15)
ax.spines[["top", "right"]].set_visible(False)
ax.tick_params(left=False)
fig.text(0.02, -0.03, "Source: Beans of Egham till data, twelve months to Oct 2026.",
         fontsize=8, color="#666666")
plt.show()

### Chart 2: The truncated bar

> **Verdict: C fail.** The y-axis starts at £4.90, so Egham High St's bar looks four times as tall as Windsor Yard's, for a spread of under 2%. Bars show value as *length*, and cutting the baseline makes the length misleading. Rebuild it with bars from zero, and state in the title that spend per visit is flat.

An investor "sees" that Windsor Yard underperforms and asks why. The founder then defends a difference of nine pence. Once an investor notices that the axis does not start at zero, they may distrust the rest of the deck. The flagships are *consistent*, which is good news for a chain about to open eight sites.

**The direction:** *"Draw the same bars from zero, with values labelled on the bars. Label the y-axis 'Average transaction (£)' and turn the top and right spines off. Add a title that makes the consistency the point, with the spread computed from the data."*

In [ ]:
# Rebuild 2: The zero baseline
spread = (atv.max() - atv.min()) / atv.min()

fig, ax = plt.subplots(figsize=(8, 4.5))
bars = ax.bar(atv.index, atv, color="#0072B2")
ax.bar_label(bars, fmt="£{:.2f}", padding=3)

ax.set_ylim(0, 6)
ax.set_title(f"Spend per visit is flat across the flagships, all within {spread:.0%}",
             loc="left", fontweight="bold")
ax.set_ylabel("Average transaction (£)")
ax.tick_params(axis="x", rotation=20)
ax.spines[["top", "right"]].set_visible(False)
fig.text(0.02, -0.08, "Source: Beans of Egham till data, five flagship sites, twelve months to Oct 2026.",
         fontsize=8, color="#666666")
plt.show()

**Aside:** You can also use [LaTeX](https://www.latex-project.org/) in your Markdown cells. This is useful if you wish to present formulas. For example, I have taken `spread = (atv.max() - atv.min()) / atv.min()` and converted it into a mathematical formula:

$$
\text{spread} = \frac{\max(\texttt{atv}) - \min(\texttt{atv})}{\min(\texttt{atv})}
$$

LaTeX is a typesetting system. It's very precise, but, alas, not very easy to read:

```
$$
\text{spread} = \frac{\max(\texttt{atv}) - \min(\texttt{atv})}{\min(\texttt{atv})}
$$
```

In any case, I would like you to know that it exists. And, for those interested, try to figure out how to use it in your own notebooks. Perhaps ask your Associate for help?

### Chart 3: The rainbow line chart

> **Verdict: R and A fail.** Rainbow lines plus a legend make the reader look back and forth between the key and the data. The finding the reader needs is about *one* line. The rainbow implies an order that does not exist. It also fails colour-blind readers. Rebuild it with grey context, one accent, and direct labels.

Iced drinks end the year at almost four times their starting level. The rainbow hides that among the other, flatter lines.

**The direction:** *"Redraw with all categories in grey except Iced drinks in #0072B2. Label every line directly at its right-hand end, no legend. Label the y-axis 'Revenue (£k/month)' and start it at zero. Add a takeaway title with the iced growth computed from the data."*

In [ ]:
# Rebuild 3: One accent line, the rest in grey
iced = monthly_cat["Iced drinks"]
nudge = {"Tea": -1.4}   # keep the two closest end labels apart

fig, ax = plt.subplots(figsize=(9, 4.5))
for cat in monthly_cat.columns.drop("Iced drinks"):
    ax.plot(monthly_cat.index, monthly_cat[cat], color="#949494", linewidth=1.5)
    ax.text(11.3, monthly_cat[cat].iloc[-1] + nudge.get(cat, 0), cat,
            color="#767676", fontsize=8, va="center")
ax.plot(monthly_cat.index, iced, color="#0072B2", linewidth=2.5)
ax.text(11.3, iced.iloc[-1], "Iced drinks", color="#0072B2",
        fontweight="bold", fontsize=9, va="center")

ax.set_title(f"Iced drinks grew from £{iced.iloc[0]:.0f}k to "
             f"£{iced.iloc[-1]:.0f}k a month in a year",
             loc="left", fontweight="bold")
ax.set_ylabel("Revenue (£k/month)")
ax.set_ylim(0, None)
ax.set_xlim(-0.5, 14.5)
ax.spines[["top", "right"]].set_visible(False)
fig.text(0.02, -0.03, "Source: Beans of Egham till data, twelve months to Oct 2026.",
         fontsize=8, color="#666666")
plt.show()

### Chart 4: The dual-axis "correlation"

> **Verdict: C fail (the most serious fault in the deck).** With two independent y-axes, the person who sets the scales can make any two lines appear to move together. The title "Marketing Spend Drives Revenue" claims a cause, and only the choice of scales supports that claim. Rebuild the chart by indexing both series to 100 on one axis. Then draw the conclusion from the rebuilt chart.

Indexed to a common start, paid social more than *doubles* over the year, while revenue rises by about a sixth. Chart 3 points to a likelier cause, the summer: Iced drinks peak in summer, and so does paid social. A season that lifts revenue can also pull the marketing budget up with it. The data supports only this conclusion: Both rose, and nothing here shows that one caused the other. The [DataViz design guide](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/dataviz-design-guide.md) gives another option, two stacked panels that share the x-axis. The [Red Flags](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-red-flags) answer below draws it.

**The direction:** *"Drop the twin axis. Index both series to 100 at the first month and plot on one axis. Use direct labels and no legend. Add a title that states both growth rates, computed from the data."*

The cell below divides each series by its first month, multiplies by 100, and plots both on one axis.

In [ ]:
# Rebuild 4: Both series indexed to 100, on one axis
indexed = pd.DataFrame({"Revenue": monthly_cat.sum(axis=1), "Paid social": marketing})
indexed = indexed / indexed.iloc[0] * 100

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(indexed.index, indexed["Paid social"], color="black", linewidth=2)
ax.plot(indexed.index, indexed["Revenue"], color="#0072B2", linewidth=2.5)
ax.text(11.3, indexed["Paid social"].iloc[-1], "Paid social", color="black", va="center")
ax.text(11.3, indexed["Revenue"].iloc[-1], "Revenue", color="#0072B2",
        fontweight="bold", va="center")

ax.set_title(f"Paid social is up {indexed['Paid social'].iloc[-1] - 100:.0f}% in a year, "
             f"revenue up {indexed['Revenue'].iloc[-1] - 100:.0f}%",
             loc="left", fontweight="bold")
ax.set_ylabel("Index (Nov 2025 = 100)")
ax.set_xlim(-0.5, 14.5)
ax.axhline(100, color="#d0d0d0", linewidth=1, zorder=0)
ax.spines[["top", "right"]].set_visible(False)
fig.text(0.02, -0.03, "Source: Beans of Egham till data and paid-social spend, twelve months to Oct 2026.",
         fontsize=8, color="#666666")
plt.show()

### Chart 5: The 3D bars

> **Verdict: C fail (with T problems).** The depth axis spreads the years out in perspective. Every bar is then hard to read against the value axis. Near rows also hide far ones. Rebuild it as one line across the quarters, in time order.

Flattened, the quarters show that revenue grows every year, with a peak every summer. The year-on-year growth rate belongs in the title, computed from the actual quarters only. The last quarter is the founder's forecast, so the figure needs a caption stating this before it goes in the deck. The rebuild also draws that quarter dashed.

**The direction:** *"Flatten to a single line of quarterly revenue across all twelve quarters in time order. Mark the Q3 peaks. Label the y-axis 'Revenue (£k/quarter)' and start it at zero. Draw the forecast quarter dashed, with a caption that states it is a forecast. Compute average year-on-year growth for the title from the actual quarters only. Add a source line."*

In [ ]:
# Rebuild 5: The quarters flattened into one line
flat = quarterly.T.stack()                       # (year, quarter) in time order
labels = [f"{q}\n{y}" for y, q in flat.index]
# Q4 2026 is a forecast, so the growth rate compares only the actual Q1 to Q3 of 2024 and 2026
actual = quarterly.loc[["Q1", "Q2", "Q3"]].sum()
growth = (actual["2026"] / actual["2024"]) ** 0.5 - 1

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(range(11), flat.iloc[:11], color="#0072B2", marker="o", linewidth=2)
ax.plot([10, 11], flat.iloc[10:], color="#0072B2", linestyle="--", linewidth=2)   # the forecast
ax.plot(11, flat.iloc[11], color="#0072B2", marker="o", markerfacecolor="white")
for i in [2, 6, 10]:                             # the Q3 peaks
    ax.plot(i, flat.iloc[i], marker="o", color="#D55E00", zorder=3)
    ax.annotate("Q3 peak", (i, flat.iloc[i]), textcoords="offset points",
                xytext=(0, 10), ha="center", fontsize=8, color="#333333")

ax.set_title(f"Revenue has grown about {growth:.0%} a year (Q1 to Q3), with a peak every summer",
             loc="left", fontweight="bold")
ax.set_xticks(range(12), labels, fontsize=8)
ax.set_ylabel("Revenue (£k/quarter)")
ax.set_ylim(0, flat.max() * 1.18)
ax.spines[["top", "right"]].set_visible(False)
fig.text(0.02, -0.06, "Dashed: Q4 2026 is the founder's forecast. "
         "The growth rate leaves it out.", fontsize=9, color="#333333")
fig.text(0.02, -0.11, "Source: Beans of Egham till data, Q1 2024 to Q3 2026, and the founder's Q4 2026 forecast.",
         fontsize=8, color="#666666")
plt.show()

### Chart 6: The crowded line chart

> **Verdict: T and R fail.** A line for every site in one panel, with a legend entry for each, is too crowded to read. One site has lost two-fifths of its trade, which is hard to see among the other lines. Rebuild it as small multiples with shared axes and an accent on the outlier.

Past about four lines, the [design guide](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/dataviz-design-guide.md) advises switching to small multiples. Each site gets its own mini chart on **shared axes**, with the all-site average as a grey reference in each. Your Associate often gives each panel its own range, which can make a weak site look like the others. Eleven panels hold steady, and Feltham Green falls away through the second half of the year.

**The direction:** *"Draw small multiples in a 3×4 grid with shared x and y axes. Each panel shows one site in #0072B2 over a grey all-site average line. Highlight the declining site in vermillion. Add an overall takeaway title with its decline computed from the data."*

One `plt.subplots(3, 4, sharex=True, sharey=True)` call draws the whole grid below. [Arranging multiple Axes](https://matplotlib.org/stable/users/explain/axes/arranging_axes.html) explains the options.

In [ ]:
# Rebuild 6: Small multiples, one panel for each site
mean_all = weekly_sites.mean(axis=1)
fg = weekly_sites["Feltham Green"]
drop = 1 - fg.tail(4).mean() / fg.head(4).mean()

fig, axes = plt.subplots(3, 4, figsize=(11, 6.5), sharex=True, sharey=True)
for ax, site in zip(axes.ravel(), weekly_sites.columns):
    accent = site == "Feltham Green"
    ax.plot(weekly_sites.index, mean_all, color="#949494", linewidth=1)
    ax.plot(weekly_sites.index, weekly_sites[site],
            color="#D55E00" if accent else "#0072B2", linewidth=1.4)
    ax.set_title(site, fontsize=9, fontweight="bold" if accent else "normal")
    ax.spines[["top", "right"]].set_visible(False)

fig.suptitle(f"Eleven sites hold steady; Feltham Green has lost about {drop:.0%} "
             "of its weekly trade over the past year (grey = all-site average)",
             x=0.02, ha="left", fontweight="bold")
fig.supxlabel("Trading week (52 weeks to Oct 2026)", fontsize=9)
fig.supylabel("Transactions per week", fontsize=9)
plt.tight_layout()
fig.text(0.02, -0.03, "Source: Beans of Egham till data, all twelve sites, 52 trading weeks to Oct 2026.",
         fontsize=8, color="#666666")
plt.show()

## 🚩 Red Flags: Worked answer

The Associate's "fix" for chart 2 restores the zero baseline, and then **adds a dual y-axis**. That is the fault you diagnosed on chart 4, in a different form.

In business terms, the overlay's two scales are unrelated. The points where the line crosses the bars, or rises above them, mean nothing. An investor reads those crossings as findings, and asks why Runnymede Park's volume sits so far below its bar. The fix has added a pattern that the data does not show, in a deck you sign. "The Associate fixed what I asked" is no defence, because the Associate is not in the room when the founder pitches on Friday.

If volume matters, use the other replacement for a dual axis in the [design guide](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/dataviz-design-guide.md): **Two stacked panels that share an x-axis**. Each panel has its own zero baseline and its own takeaway title.

In [ ]:
# Deck chart 2's overlay, corrected: Two panels, each with its own zero baseline
flagship_volume = weekly_sites[FLAGSHIPS].mean()

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(8, 6.5), sharex=True)

bars1 = ax1.bar(atv.index, atv, color="#0072B2")
ax1.bar_label(bars1, fmt="£{:.2f}", padding=3, fontsize=8)
ax1.set_ylim(0, 6)
ax1.set_ylabel("Avg transaction (£)")
ax1.set_title("Spend per visit is flat across the flagships…", loc="left", fontweight="bold")
ax1.spines[["top", "right"]].set_visible(False)

bars2 = ax2.bar(atv.index, flagship_volume, color="#D55E00")
ax2.bar_label(bars2, fmt="{:,.0f}", padding=3, fontsize=8)
ax2.set_ylim(0, flagship_volume.max() * 1.25)
ax2.set_ylabel("Transactions per week")
ax2.set_title("…so footfall is what separates them", loc="left", fontweight="bold")
ax2.tick_params(axis="x", rotation=20)
ax2.spines[["top", "right"]].set_visible(False)

plt.tight_layout()
fig.text(0.02, -0.03, "Source: Beans of Egham till data, five flagship sites, twelve months to Oct 2026.",
         fontsize=8, color="#666666")
plt.show()

The flagships take about the same per visit, so footfall decides which of them earns most.

## The memo to Fairweather (model answer)

> **To:** N. Fairweather · **From:** [Junior Analyst](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-analyst), Runnymede · **Re:** Deck figures: The habits behind the faults
>
> The numbers are all sound. They make a strong case for the pitch. The problems come from a few habits in how the charts were made. The first is **choosing a chart from what the data can plot**. The pie, the 3D bars, and the all-sites line chart each plot all the data, whatever the reader needs to see. Start from what the reader must compare (a rank, a trend, or a relationship), and choose the chart that shows it. The second is **axes that suggest a pattern the data does not show**. The truncated baseline makes a small gap look large. The twin y-axes suggest a correlation. Start every bar chart at zero. Put the two series on one indexed scale, or in two panels. The third is **decoration that makes the finding hard to see**. The deck uses rainbow palettes, legends, and perspective effects. One accent colour, grey for the rest, and direct labels would show the reader where to look. A TRACE pass on every key figure before it goes out catches all of these. I am happy to walk you through it over coffee.

## 🤖 AI failure gallery

Each failure is your Associate *working as designed*. It reproduces tutorial code, which is written to show off a library's features. The problems come from library defaults, which look reasonable until you check them against TRACE.

### 1. Rainbow palettes by default

**The plausible output:** Suppose you ask for "monthly revenue by category, make the lines easy to tell apart". You often get `palette="rainbow"`, `cmap="jet"`, or a hue for each line picked to be as different as possible, plus a legend.

**Why it is wrong in business terms:** Rainbow hues imply an order that categories do not have ("is Tea *more than* Food?"). Rainbow hues also fail colour-blind readers, who may not be able to tell some of the lines apart. Section "4. A: Accessible" of the [design guide](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/dataviz-design-guide.md) explains how common that is. The rainbow spreads the reader's attention across every series when the argument is about one.

**The catch:** When you see more than four hues, ask *"what does colour mean here?"* If the answer is "which line is which", direct labels do that job better. Ask for grey plus one accent, and name the palette (*"Okabe-Ito blue #0072B2, rest in grey"*).

### 2. The legend habit

**The plausible output:** Your Associate adds `ax.legend()` to almost everything. In this week's Demonstration, that includes a legend for bars whose names are already on the axis.

**Why it is wrong in business terms:** A legend makes the reader match a floating key to the data. At best, that slows the reader down. Printed in greyscale, a colour key cannot be matched to the lines at all. A redundant legend also signals template code.

**The catch:** Check whether you could still read the chart without its legend. If you could, remove the legend. If you could not, ask for direct labels, with values on bars (`ax.bar_label`) and names at line ends (`ax.text`). Your Associate does both well when you ask.

### 3. Decorative 3D

**The plausible output:** Ask for a chart that "looks impressive for a pitch deck", and you may get 3D bars. Your Associate learned from many tutorials that treat the third dimension as polish.

**Why it is wrong in business terms:** Perspective distorts the one quantity a bar exists to show, its length. Chart 5 above shows the effect.

**The catch:** Ask whether the third dimension shows anything a flat chart cannot. If it does not, flatten the chart.

### 4. Titles that name variables, and axes that name nothing

**The plausible output:** A title such as `ax.set_title("Revenue by Branch")`, an unlabelled y-axis, no units, and no source. Your Associate labels a chart with its *inputs*, because your prompt named the inputs and not the finding.

**Why it is wrong in business terms:** Without units, the reader cannot tell whether an axis shows £, £k, or a weekly figure. A title that names a variable leaves each reader to draw their own conclusion. A business figure should state that conclusion for them. An unsourced, unlabelled figure cannot be checked, so a reader cannot use it as evidence.

**The catch:** Read every axis aloud with its unit. If you cannot say the unit, add it to the label. You decide what the takeaway title states. Compute any number in it from the data (with an f-string), state the finding, and add a source line.

---

*Run TRACE yourself on every chart your Associate draws, to catch these failures.*